<a href="https://colab.research.google.com/github/Matlosa11/Matlosa_Repository/blob/main/Matlosa_Week5_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Target Variable and Input Features

### Target

The target variable is `event_type`.

I will convert it into a binary classification target:

- 1 = Breakdown
- 0 = Non-breakdown event

### Input Features

The following features will be used:

- `equipment_type`: The type of mining equipment.
- `manufacturer`: The equipment manufacturer.
- `model_year`: The year the equipment model was manufactured.
- `mine_type`: The mining environment.
- `operating_hours`: The recorded operating hours.
- `days_since_last_service`: The number of days since the last service.
- `was_scheduled`: Whether the event was scheduled.
- `oil_analysis_flag`: Whether an oil analysis warning was recorded.
- `vibration_flag`: Whether a vibration warning was recorded.
- `temperature_flag`: Whether a temperature warning was recorded.

These features represent equipment characteristics, usage, maintenance status, and condition-monitoring indicators that may help distinguish breakdown events from other events.

In [2]:
# Import the libraries
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load the mining equipment dataset
dataset = load_dataset(
    "electricsheepafrica/africa-synth-mining-equipment-failure-all"
)

# Convert the dataset to a Pandas DataFrame
split_name = next(iter(dataset))
df = dataset[split_name].to_pandas()

# Display the first five records
display(df.head())

# Display the available columns
print("Dataset columns:")
print(df.columns.tolist())

print("\nDataset shape:", df.shape)

,record_id,equipment_id,equipment_type,manufacturer,model_year,mine_type,country,operating_hours,event_date,event_type,...,severity,downtime_hours,repair_cost_usd,production_loss_tonnes,maintenance_type,was_scheduled,days_since_last_service,oil_analysis_flag,vibration_flag,temperature_flag
0,EQP-ADB3590D,UNIT-1803,haul_truck,Caterpillar,2008,underground,drc,64584,2022-12-03,breakdown,...,major,17.4,70862.60,1711.0,corrective,False,27,False,False,False
1,EQP-596EE017,UNIT-7964,crusher,Sandvik,2019,open_pit,botswana,19253,2023-08-03,breakdown,...,major,46.3,74513.34,10010.0,corrective,False,57,True,False,False
2,EQP-23341C24,UNIT-6890,pump,Metso,2011,open_pit,south_africa,52894,2020-05-28,breakdown,...,major,10.7,17661.80,580.0,corrective,False,107,False,False,False
3,EQP-D8EF5309,UNIT-4949,loader,Volvo,2019,processing_plant,zambia,18369,2024-11-21,condition_based,...,routine,2.5,2218.70,240.0,predictive,False,23,False,True,False
4,EQP-5A611650,UNIT-4896,excavator,Hitachi,2017,underground,ghana,27901,2022-12-30,condition_based,...,minor,7.7,36773.60,1468.0,predictive,False,57,False,False,False


Dataset columns:
['record_id', 'equipment_id', 'equipment_type', 'manufacturer', 'model_year', 'mine_type', 'country', 'operating_hours', 'event_date', 'event_type', 'failure_mode', 'component_failed', 'severity', 'downtime_hours', 'repair_cost_usd', 'production_loss_tonnes', 'maintenance_type', 'was_scheduled', 'days_since_last_service', 'oil_analysis_flag', 'vibration_flag', 'temperature_flag']

Dataset shape: (5000, 22)


In [3]:
# Select the features and target
features = [
    "equipment_type",
    "manufacturer",
    "model_year",
    "mine_type",
    "operating_hours",
    "days_since_last_service",
    "was_scheduled",
    "oil_analysis_flag",
    "vibration_flag",
    "temperature_flag"
]

target = "event_type"

# Check that all required columns exist
required_columns = features + [target]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"These columns are missing from the dataset: {missing_columns}"
    )

# Keep the required columns and remove incomplete records
data = df[required_columns].copy()
data = data.dropna()

# Create the binary target:
# 1 means breakdown; 0 means another event type
data["breakdown"] = (
    data[target].astype(str).str.lower() == "breakdown"
).astype(int)

# Separate the input features and target
X = data[features]
y = data["breakdown"]

# Display the number of records in each class
print("Target class distribution:")
print(y.value_counts())

print("\nTarget class percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

Target class distribution:
breakdown
0    3458
1    1542
Name: count, dtype: int64

Target class percentages:
breakdown
0    69.16
1    30.84
Name: proportion, dtype: float64


In [ ]:
## Type of Machine Learning Problem

This is a supervised binary classification problem.

It is supervised because the dataset contains recorded event types that can be used as labels during model training.

It is a classification problem because the objective is to assign each equipment event to one of two categories:

- Breakdown
- Non-breakdown

The model will learn patterns from the training data and use them to predict the category of previously unseen records.

Regression would be appropriate if the objective were to predict a continuous numerical value, such as the number of downtime hours or production losses in tonnes. However, the current objective is to predict a category.

In [4]:
from sklearn.model_selection import train_test_split

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training records: 4000
Testing records: 1000

Training target distribution:
breakdown
0    0.692
1    0.308
Name: proportion, dtype: float64

Testing target distribution:
breakdown
0    0.692
1    0.308
Name: proportion, dtype: float64


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Identify numerical and categorical features
numeric_features = [
    "model_year",
    "operating_hours",
    "days_since_last_service"
]

categorical_features = [
    "equipment_type",
    "manufacturer",
    "mine_type",
    "was_scheduled",
    "oil_analysis_flag",
    "vibration_flag",
    "temperature_flag"
]

# Preprocess numerical features
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocess categorical features
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Apply the appropriate preprocessing to each feature type
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

# Model 1: Logistic Regression
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Model 2: Random Forest
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ))
])

print("Both models have been defined.")

Both models have been defined.


In [6]:
# Train Logistic Regression
logistic_model.fit(X_train, y_train)

# Train Random Forest
random_forest_model.fit(X_train, y_train)

print("Both models have been trained successfully.")

Both models have been trained successfully.


In [7]:
# Predict the test set using both models
logistic_predictions = logistic_model.predict(X_test)

random_forest_predictions = random_forest_model.predict(X_test)

# Display a sample of actual and predicted results
comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Logistic Regression": logistic_predictions,
    "Random Forest": random_forest_predictions
})

display(comparison.head(10))

,Actual,Logistic Regression,Random Forest
0,0,0,0
1,1,1,1
2,0,0,0
3,1,1,0
4,0,0,0
5,0,0,0
6,0,0,0
7,0,0,0
8,0,0,0
9,0,0,0


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Store the predictions
predictions = {
    "Logistic Regression": logistic_predictions,
    "Random Forest": random_forest_predictions
}

# Calculate performance metrics
results = []

for model_name, prediction in predictions.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(
            y_test, prediction, zero_division=0
        ),
        "Recall": recall_score(
            y_test, prediction, zero_division=0
        ),
        "F1-score": f1_score(
            y_test, prediction, zero_division=0
        )
    })

results_df = pd.DataFrame(results)

display(results_df.round(3))

,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.848,0.671,0.994,0.801
1,Random Forest,0.816,0.670,0.792,0.726


In [9]:
# Display the confusion matrix and classification report for each model
for model_name, prediction in predictions.items():
    print("\n", model_name)
    print("-" * 40)

    print("Confusion matrix:")
    print(confusion_matrix(y_test, prediction))

    print("\nClassification report:")
    print(classification_report(
        y_test,
        prediction,
        target_names=["Non-breakdown", "Breakdown"],
        zero_division=0
    ))


 Logistic Regression
----------------------------------------
Confusion matrix:
[[542 150]
 [  2 306]]

Classification report:
               precision    recall  f1-score   support

Non-breakdown       1.00      0.78      0.88       692
    Breakdown       0.67      0.99      0.80       308

     accuracy                           0.85      1000
    macro avg       0.83      0.89      0.84      1000
 weighted avg       0.90      0.85      0.85      1000


 Random Forest
----------------------------------------
Confusion matrix:
[[572 120]
 [ 64 244]]

Classification report:
               precision    recall  f1-score   support

Non-breakdown       0.90      0.83      0.86       692
    Breakdown       0.67      0.79      0.73       308

     accuracy                           0.82      1000
    macro avg       0.78      0.81      0.79      1000
 weighted avg       0.83      0.82      0.82      1000



In [ ]:
## Model Performance Comparison

The performance of Logistic Regression and Random Forest was evaluated using accuracy, precision, recall, and F1-score on the same test dataset.

Accuracy indicates the overall proportion of correct predictions. Precision indicates how reliable the model's breakdown predictions are, while recall indicates how effectively it identifies actual breakdown events. The F1-score balances precision and recall.

The model with the higher F1-score provides a better balance between precision and recall on this test set. However, the preferred model also depends on the consequences of missed breakdowns and false alarms.

The results should be interpreted cautiously because the dataset is synthetic. Good performance on this dataset does not demonstrate that either model can reliably predict breakdowns in a real underground mine.